# HuggingFace BlenderBot Chatbot

Fresh Python 3.11+ CPU kernel. Downloads pinned official BlenderBot400M weights; no API key. Transformers4.41.2 intentionally retains the source Conversation/conversational API; newer releases remove it. Explicit official model-specific template, deterministic generation and complete original three-turn history. Add the omitted second-turn inference call; replace unavailable local checkpoint path. This is a historical API compatibility example, not a claim of current production recommendation or persistent model memory. Exact original code/transcripts remain in appendices.

In [ ]:
import os
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "torch==2.10.0", "transformers==4.41.2", "numpy==2.3.5", "certifi==2026.10.23", "ipywidgets==8.1.7"], check=True)
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
import certifi
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
import numpy as np
import torch
import transformers
torch.set_num_threads(2)
torch.manual_seed(42)
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| Transformers:", transformers.__version__)

## Example 0

In [ ]:
print("Pinned Transformers Conversation API and PyTorch installed in the active kernel.")

## Example 1

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
print("Pipeline, tokenizer and model classes imported.")

## Example 2

In [ ]:
MODEL_ID = "facebook/blenderbot-400M-distill"
REVISION = "eaaf64e3be20ad1f1fb0bdf689565ba52c97eafe"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, token=False)
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}{{ ' ' }}{% endif %}"
    "{{ message['content'] }}"
    "{% if not loop.last %}{{ '  ' }}{% endif %}"
    "{% endfor %}{{ eos_token }}"
)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, revision=REVISION, token=False).to("cpu").eval()
generation_options = dict(max_new_tokens=64, do_sample=False)
chatbot = pipeline(task="conversational", model=model, tokenizer=tokenizer, device=-1, **generation_options)
print("Loaded:", MODEL_ID, "| revision:", REVISION)
print("Context limit:", model.config.max_position_embeddings)

## Example 3

In [ ]:
from transformers import Conversation
from uuid import UUID
user_message = """
What are the best ways to learn AI?
"""
conversation = Conversation(user_message, conversation_id=UUID("00000000-0000-0000-0000-000000000004"))
print(conversation)

## Example 4

In [ ]:
conversation = chatbot(conversation)
first_snapshot = [dict(message) for message in conversation.messages]
print(conversation)

## Example 5

In [ ]:
conversation.add_message({"role": "user", "content": """
What else do you recommend?
"""})
conversation = chatbot(conversation)
second_snapshot = [dict(message) for message in conversation.messages]
print(conversation)

## Example 6

In [ ]:
conversation.add_message({"role": "user", "content": """
I want to learn about machine learning?
"""})
conversation = chatbot(conversation)
print(conversation)

## Independent regression checks

In [ ]:
assert not model.training
messages = conversation.messages
assert [m["role"] for m in messages] == ["user", "assistant"] * 3
assert [m["content"].strip() for m in messages[::2]] == ["What are the best ways to learn AI?", "What else do you recommend?", "I want to learn about machine learning?"]
assert messages[:2] == first_snapshot and messages[:4] == second_snapshot
assert all(m["content"].strip() for m in messages)
for end in [1, 3, 5]:
    encoded = tokenizer.apply_chat_template(messages[:end], add_generation_prompt=True, return_tensors="pt")
    assert encoded.shape[1] <= model.config.max_position_embeddings
    with torch.inference_mode():
        result = model.generate(encoded, **generation_options)
    decoded = tokenizer.decode(result[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    assert decoded == messages[end]["content"]
    print("Turn", (end+1)//2, ": direct generation matches conversation pipeline; context tokens:", encoded.shape[1])
assert all(p.grad is None for p in model.parameters())
print("All three actual model replies, six-message role order, immutable history prefixes, context limits and direct-generation checks passed.")

## Original source appendix

Full original blocks from raw source SHA-256 4a2276ba09ce02eb8a47c04a02398ffa925d285748c2aa743e99d886bc0b03f1. Preserved verbatim as nonexecuting historical material, including mistakes/legacy setup alternatives.

### Original block 1

```text
!pip install transformers
```

### Original block 2

```text
from transformers import pipeline
```

### Original block 3

```text
chatbot = pipeline(task="conversational",
                   model="./models/facebook/blenderbot-400M-distill")
```

### Original block 4

```text
from transformers import Conversation
user_message = """
What are the best ways to learn AI?
"""
conversation = Conversation(user_message)
print(conversation)
```

### Original block 5

```text
conversation = chatbot(conversation)
print(conversation)
```

### Original block 6

```text
conversation.add_message(
    {"role": "user",
     "content": """
What else do you recommend?
"""
    })
```

### Original block 7

```text
conversation.add_message(
    {"role": "user",
     "content": """
I want to learn about machine learning?
"""
    })

conversation = chatbot(conversation)

print(conversation)
```

## Original source outputs

Conversation id: 9dd48543–5070–4742–911c-28861df64ece user: What are the best ways to learn AI?

Conversation id: 9dd48543–5070–4742–911c-28861df64ece user: What are the best ways to learn AI?

assistant: I think the best way to learn about it is to read about it on the internet.

Conversation id: 9dd48543–5070–4742–911c-28861df64ece user: What are the best ways to learn AI?

assistant: I think the best way to learn about it is to read about it on the internet. user: What else do you recommend?

assistant: There are so many things to learn. It’s hard to pick just one. What do you want to know about it?

Conversation id: 9dd48543–5070–4742–911c-28861df64ece user: What are the best ways to learn AI?

assistant: I think the best way to learn about it is to read about it on the internet. user: What else do you recommend?

assistant: There are so many things to learn. It’s hard to pick just one. What do you want to know about it? user: I want to learn about machine learning?

assistant: Do you have any hobbies? I like to play video games. Do you like video games?